
### Diabetes Prediction - Ensemble Models vs Results from the Paper's Logistic Regression Baseline



In [ ]:
import warnings
from xml.sax.handler import all_features

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score,
    classification_report
)

%matplotlib inline


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (GradientBoostingClassifier, ExtraTreesClassifier)
from sklearn.naive_bayes import GaussianNB

In [ ]:
# Load the dataset
dbdf = pd.read_csv('Diabetic.csv')
dbdf_copy = dbdf.copy()
# Display the first few rows of the dataset
print(dbdf.head())

### EDA

 Critical risk: The EDA fails to detect biologically implausible zeros, which act as de facto missing values.

 Impact: Models trained on raw zeros will learn incorrect patterns (e.g., "low glucose = no diabetes").
 Action: Replace zeros with medians or use domain-specific imputation (e.g., clinical norms).
 Preprocessing decision: Must explicitly handle zeros in Glucose, Insulin, BloodPressure, SkinThickness, BMI.


In [ ]:

# Check for missing values
print(dbdf.isnull().sum())
# Fill missing values with the median of each column
dbdf.fillna(dbdf.median(), inplace=True)
# Check for class imbalance
print(dbdf['Outcome'].value_counts())


In [ ]:
#STEP 1: Median Imputation (paper replaces zeros with medians)
cols_with_zeros = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

for col in cols_with_zeros:
    median_val = dbdf[col].replace(0, np.nan).median()
    dbdf[col] = dbdf[col].replace(0, median_val)

In [ ]:
# Visualize class distribution
plt.figure(figsize=(8, 6))
plt.bar(dbdf['Outcome'].value_counts().index, dbdf['Outcome'].value_counts().values)
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.title('Class Distribution')
plt.show()

# Observations:
Clearly there is a class imbalance in the dataset, with more negative cases (Outcome=0) than positive cases (Outcome=1). This imbalance can affect model performance, especially for metrics like accuracy, which may be misleading. We will need to consider this when evaluating our models and may want to use metrics like precision, recall, and F1-score in addition to accuracy.

In [ ]:

axes = dbdf.hist(
    sharex=False,
    sharey=False,
    figsize=(12,10),   # increase figure size
    bins=30            # more intervals
)

for ax in axes.flatten():
    ax.set_title(ax.get_title(),fontsize=12)              # remove top labels (column names)
    ax.set_xlabel("Value", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)
    ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.show()

## Observations:
- The features have varying distributions, with some showing skewness (e.g., Glucose, Insulin) and others being more normally distributed (e.g., Age).
- The presence of outliers is evident in some features(insulin, skinThickness), which may need to be addressed during preprocessing.
- The class imbalance is visually confirmed, with a higher frequency of negative cases (Outcome=0) compared to positive cases (Outcome=1).

In [ ]:
## TODO Quantitive measures (skewness/ kurtosis)

print(dbdf.skew())  # Check skewness
print(dbdf.describe().loc[['min', 'max']])  # Check ranges



## Observations :

Apply RobustScaler scaling for skewed features

In [ ]:
axes = dbdf.plot(
    kind='density',
    subplots=True,
    layout=(3,3),
    figsize=(12,10),
    sharex=False,
    sharey=False
    ,bw_method=.2
)

for ax in axes.flatten():
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.tick_params(axis='both', labelsize=8)
    ax.set_title(ax.get_title(),fontsize=12)

plt.tight_layout()
plt.show()

## Observations:
- The density plots confirm the skewness in features like Glucose and Insulin, which may benefit from transformations (e.g., log transformation) to improve model performance.
- The Age feature appears to have a more normal distribution, which may not require transformation.
- The class imbalance is again visually evident, reinforcing the need to consider appropriate evaluation metrics.
- Overall, the EDA suggests that preprocessing steps such as handling outliers, addressing skewness, and considering class imbalance will be important for building effective models on this dataset.

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,8))
sns.heatmap(dbdf.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix (Paper Reproduction)")
plt.show()

In [ ]:
## Observations:



In [ ]:
plt.figure(figsize=(8,6))

scatter = plt.scatter(
    dbdf['Glucose'],
    dbdf['Age'],
    c=dbdf['Outcome'],
    cmap='coolwarm'
)

plt.xlabel("Glucose")
plt.ylabel("Age")
plt.title("Interaction: Glucose vs Age")
plt.colorbar(label='Diabetes(1)')
plt.show()


## Observations:
Higher Glucose + Older Age → Higher Diabetes Risk:

Red points (diabetic) cluster in the top-right (high Glucose, high Age).

Non-linear relationship:

Risk increases sharply at higher Glucose levels (e.g., >140 mg/dL, the clinical threshold for diabetes).
Age has a gradual effect but is more impactful at higher Glucose levels.



In [ ]:
## PAPER REPRODUCTION: Diabetes Prediction

In [ ]:
# Feature engineering and selection
dbdf['Glucose_Age'] = dbdf['Glucose'] * dbdf['Age']

In [ ]:
all_features=['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Glucose_Age']

In [ ]:
seed=42
test_size=.3
prob_threshold =.35
model_metrics_df=pd.DataFrame()
tuning_score="average_precision"

In [ ]:
X=dbdf[all_features]
y=dbdf['Outcome']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=test_size,random_state=seed,stratify=y)

In [ ]:
#Apply RobustScaler(fit for train only)
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

A random forest classifier is used to derive the important features from the dataset. The feature importance scores are calculated based on the mean decrease in impurity (Gini importance) for each feature across all trees in the random forest. The top 5 features with the highest importance scores are selected for further analysis and model building. This process helps to identify which features contribute most to the prediction of diabetes, allowing for a more focused and potentially more accurate model.


In [ ]:
rfc=RandomForestClassifier(n_estimators=12)
#Observe , it is still y_train
rfc.fit(X_train_scaled,y_train)

In [ ]:
# RF Feature Importance

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': rfc.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8,5))
colors = ['red' if f in ['Glucose','BMI','Age',
                         'DiabetesPedigreeFunction','Pregnancies','Glucose_Age']
          else 'steelblue' for f in importance_df['Feature']]
plt.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
plt.title("RFC Feature Importance\n(Red = Paper's 5 Key Predictors)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

print("Paper's 5 predictors confirmed by RFC ranking:")
print(importance_df.to_string(index=False))



### Observations:
Clearly the paper's 5 key predictors (Glucose, BMI, Age, DiabetesPedigreeFunction, Pregnancies) are among the top features identified by the random forest classifier, confirming their importance in predicting diabetes. This validates the paper's feature selection and suggests that these features should be included in our models for better performance.

## Recommended next steps
Interaction terms:

Test if Age × Pregnancies or BMI × Glucose improve model performance.

Feature Engineering:

* Glucose categories: Bin into <100 (normal), 100–140 (prediabetes), >140 (diabetes).
* BMI categories: Underweight (<18.5), Normal (18.5–25), Overweight (25–30), Obese (>30).
* Age × Glucose interaction: Captures that glucose is more predictive in older patients.






In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

#Note default threshold to .35
#Why:Threshold = 0.35 will increase recall (fewer missed diabetes cases) at the cost of slightly lower precision.
def evaluate_model(model, X_test, y_test, model_name, threshold=prob_threshold):
    """
    Evaluate a model with a custom threshold for binary classification.

    Args:
        model: Trained model (must have predict_proba or decision_function).
        X_test: Test features.
        y_test: Test labels.
        model_name: Name of the model (for output).
        threshold: Custom threshold for positive class (default=0.5).

    Returns:
        Dictionary with metrics (Accuracy, Precision, Recall, F1, AUC, AUPRC).
    """
    y_pred = None
    y_prob = None

    # Get probabilities or decision scores
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]  # Probability of Class 1
        y_pred = (y_prob >= threshold).astype(int)   # Apply custom threshold
    elif hasattr(model, "decision_function"):
        y_decision = model.decision_function(X_test)
        # Convert decision scores to probabilities (for SVM, etc.)
        y_prob = 1 / (1 + np.exp(-y_decision))  # Sigmoid transform
        y_pred = (y_prob >= threshold).astype(int)
    else:
        # Fallback to default predict (threshold=0.5)
        y_pred = model.predict(X_test)

    # Calculate metrics
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'AUC': round(roc_auc_score(y_test, y_prob), 3) if y_prob is not None else None,
        'AUPRC': round(average_precision_score(y_test, y_prob), 3) if y_prob is not None else None,
        'Threshold': threshold  # Track the threshold used
    }

In [ ]:

dbdf_lr = dbdf.copy()

### Observations:

* Age and  pregnancies have a strong correlation


In [ ]:
# STEP 2: Five Predictors identified by paper
# Paper: glucose, pregnancy, BMI, pedigree, age and also Glucose_Age
# ------------------------------------------------------------
important_features = ['Pregnancies', 'Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age','Glucose_Age']

X_paper = dbdf_lr[important_features].copy()
y_paper = dbdf_lr['Outcome'].copy()


In [ ]:

X_train_paper, X_test_paper, y_train_paper, y_test_paper = train_test_split(
    X_paper, y_paper, test_size=test_size, random_state=seed, stratify=y
)

## Let's train a logistic regression model using the paper's 5 key predictors and evaluate its performance.

In [ ]:
#Apply RobustScaler
X_train_paper_scaled = scaler.fit_transform(X_train_paper)
X_test_paper_scaled = scaler.transform(X_test_paper)

In [ ]:
#Logistic Regression Tuning

lr_paper_param_grid = {
    'C': [0.01, 0.1, 1, 5, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'class_weight': [None, 'balanced']
}

lr_paper_grid = GridSearchCV(
    estimator=LogisticRegression(random_state=seed, max_iter=1000),
    param_grid=lr_paper_param_grid,
    cv=5,
    scoring='average_precision',
    n_jobs=-1
)

lr_paper_grid.fit(X_train_paper_scaled, y_train_paper)

print(f"Best parameters for Logistic Regression: {lr_paper_grid.best_params_}")

paper_tuned_lr = lr_paper_grid.best_estimator_

y_prob_paper = paper_tuned_lr.predict_proba(X_test_paper_scaled)[:, 1]
#Threshold = 0.35 will increase recall (fewer missed diabetes cases) at the cost of slightly lower precision.
y_pred_paper = (y_prob_paper >= prob_threshold).astype(int)

In [ ]:
from sklearn.metrics import roc_auc_score
#Logistic Metrics create comparison_df

print("Classification Report for Tuned Logistic Regression:")
res=evaluate_model(paper_tuned_lr,X_test_paper_scaled,y_test_paper,'Paper LR')
model_metrics_df=pd.DataFrame([res])



In [ ]:
model_metrics_df

### Observations

- **Accuracy (72.4%)**:
  Approximately 1 in 4 patients is misclassified. Due to class imbalance, accuracy is the least meaningful metric in this context.

- **Class 0 (Non-Diabetic)**
  - Precision: 0.81 → When the model predicts *Not Diabetic*, it is correct 81% of the time.
  - Recall: 0.75 → It correctly identifies 75% of all actual non-diabetic patients.
  - The model is noticeably better at identifying healthy patients than diabetic ones.

- **Class 1 (Diabetic)**
  - Precision: 0.60 → When the model predicts *Diabetic*, it is correct 60% of the time.
  - Recall: 0.67 → It correctly identifies 67% of actual diabetic patients.
  - The model is less effective at identifying diabetic patients, which is critical in medical diagnosis.

- **F1 Score Comparison**
  - Class 0: 0.78
  - Class 1: 0.63
  - The 15-point gap confirms the model is significantly better on the majority class.
  - This reflects the dataset imbalance (~65% vs 35%).

- **AUPRC (0.66)**
  - This is the most reliable metric for imbalanced data.
  - The model performs **better than random (~0.35 baseline)** but still has **moderate discriminative ability**.

- **Model Behavior Insight**
  - The model is **conservative** → favors predicting *Not Diabetic*.
  - This improves overall accuracy
  - But causes **33% of diabetic patients to be missed**, which is the most costly error in a screening setting.


In [ ]:
# PR Curve for Logistic Regression
precision, recall, _ = precision_recall_curve(y_test_paper, y_prob_paper)
baseline=y_test_paper.mean()

plt.figure()
plt.plot(recall, precision, label="Logistic")
plt.hlines(baseline, 0, 1, linestyles='dashed', color='gray',
           label=f'Baseline={baseline:.2f}')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR Curve")
plt.legend()
plt.show()

In [ ]:
# ROC Curve for Logistic Regression
fpr, tpr, _ = roc_curve(y_test_paper, y_prob_paper)

plt.figure()
plt.plot(fpr, tpr, label="ROC")
plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("ROC Curve")
plt.show()

In [ ]:
# Confusion Matrix for Logistic Regression
cm = confusion_matrix(y_test_paper, y_pred_paper)
plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Logistic Regression")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['No Diabetes', 'Diabetes'], rotation=45)
plt.yticks(tick_marks, ['No Diabetes', 'Diabetes'])
thresh = cm.max() / 2.
for i, j in np.ndindex(cm.shape):
    plt.text(j, i, format(cm[i, j], 'd'),
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.show()

In [ ]:
# safer baseline (median instead of mean)
baseline = np.median(X_train_paper_scaled,axis=0)

## Observations:
* The left plot shows how predicted diabetes risk increases with glucose levels for different ages. The curves are steeper at higher glucose levels, indicating a stronger effect of glucose on diabetes risk. Older ages also show higher probabilities, especially at elevated glucose levels.
* The right plot shows how predicted diabetes risk increases with age for different glucose levels. The curves are steeper at higher glucose levels, indicating that age has a stronger effect on diabetes risk when glucose levels are elevated. At lower glucose levels, the increase in risk with age is more gradual.
* Both plots confirm that glucose and age are key predictors of diabetes risk, with their effects being more pronounced at higher levels. This aligns with medical understanding that both high glucose and older age are significant risk factors for diabetes. The model captures these relationships, which is a positive sign of its clinical relevance.

### A decision tree model using the same 5 key predictors from the paper is used to validate the logistic regression results and provide a more interpretable model for clinical insights.

In [ ]:
# Decision Tree From Paper (Validator)
#Note DT only uses the paper 5 features, no Glucose_Age. Also , DTs are scale-invariant, so no need to scale.
dt_features=['Glucose','BMI','Age','DiabetesPedigreeFunction','Pregnancies']
X_tree = dbdf_lr[dt_features].copy()

# Use the same train/test indices as X_train_paper
X_tree_train = X_tree.loc[X_train_paper.index]
X_tree_test = X_tree.loc[X_test_paper.index]
y_tree_train = y_paper.loc[X_train_paper.index]  # Ensure y is aligned!
y_tree_test = y_paper.loc[X_test_paper.index]


# Decision Tree Tuning
dt_param_grid = {
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced']
}

dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=seed),
    param_grid=dt_param_grid,
    cv=5,
    scoring='average_precision',
    n_jobs=-1
)

# Now train DT with the correctly aligned y
dt_grid.fit(X_tree_train, y_tree_train)


print(f"Best parameters for Decision Tree: {dt_grid.best_params_}")

## Observations:

This is a very simple tree, which is good for interpretability and suggests that the model is not overfitting. The use of 'balanced' class weights indicates that the model is compensating for the class imbalance in the dataset, which should help improve recall for the minority class (diabetic patients). The shallow depth and small leaf size suggest that the model is capturing only the most important splits, which can enhance generalization to unseen data.


In [ ]:
tuned_tree = dt_grid.best_estimator_

tuned_tree_prob = tuned_tree.predict_proba(X_tree_test)[:, 1]
#tuned_tree_pred = tuned_tree.predict(X_tree_test)
tuned_tree_pred = (tuned_tree_prob >= prob_threshold).astype(int)

In [ ]:
print(f"Classification Report for Decision Tree:")

results = evaluate_model(tuned_tree,X_tree_test,y_test,"Validator DT",threshold=prob_threshold)

model_metrics_df=pd.concat([model_metrics_df, pd.DataFrame([results])],
                           ignore_index=True)
model_metrics_df


## Observations:
* Compare against paper LR

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(22, 12))  # ✅ larger figure

plot_tree(
    tuned_tree,
    feature_names=X.columns,
    class_names=['No DM','DM'],
    filled=True,
    rounded=True,
    fontsize=15 ,  # IMPORTANT FIX
    impurity=True, #show gini
)

plt.title("Classification Tree (Paper Reproduction)")
plt.show()

In [ ]:
# PR Curve for Decision Tree
baseline = y_test_paper.mean()
precision_dt, recall_dt, _ = precision_recall_curve(y_test_paper, tuned_tree_prob)

plt.figure()
plt.plot(recall_dt, precision_dt, label="Decision Tree")
plt.hlines(baseline, 0, 1, linestyles='dashed', color='gray', label=f'Baseline={baseline:.2f}')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR Curve - Decision Tree")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Validate Logic

lr_imp = pd.Series(paper_tuned_lr.coef_[0], index=X_paper.columns).sort_values(ascending=False)
print("Logistic Regression Feature Importances:")
print(lr_imp)

lr_top_features = set(lr_imp.head(3).index)
dt_top_features = set(X_paper.columns[tuned_tree.feature_importances_.argsort()[::-1][:3]])

common = lr_top_features & dt_top_features

print("LR top 3 features   :", lr_top_features)
print("DTree top 3 features:", dt_top_features)
print(f"Common top 3        : {common}")

## Observations:
LR ranks Pregnancies and DiabetesPedigreeFunction highest
Pregnancies (0.111) and BMI (0.102) lead in LR. This is the scaling issue noted earlier — LR coefficients reflect log-odds change per unit, so features with smaller numeric ranges appear to have larger coefficients. Glucose has a wide range (50-200) so its per-unit coefficient is small (0.038) even though it is clinically the strongest predictor.

DTree ranks Glucose and Age highest
Decision Tree splits on features that most reduce impurity regardless of scale. Glucose appearing in DTree top 3 but not LR top 3 confirms the scaling issue in LR — Glucose is genuinely important but its coefficient is suppressed by its large numeric range.

BMI is the only common top 3 feature
BMI appears in both LR top 3 (0.102) and DTree top 3. This makes BMI the most robustly validated predictor across both methods — it ranks highly regardless of how importance is measured. This aligns with the paper's finding that BMI is one of the five key predictors.

Areas to improve:

The single overlap is not a failure of the models — it reflects the fundamental difference between how LR and DTree measure importance. LR measures coefficient magnitude (scale-dependent), DTree measures impurity reduction (scale-independent). To get a fair comparison from LR you would need to standardise features first using StandardScaler before fitting. Without scaling, LR feature importance rankings should be interpreted with caution.


## Now lets create 3 different ensemble models using the same 5 key predictors and compare their performance to the paper's logistic regression.

* Hard Voting Classifier Ensemble
* Bagging ensemble.
* Stacking

In [ ]:
# Run on the features identified by the paper (for consistency)

X = dbdf_lr[important_features].copy()
y = dbdf_lr['Outcome'].copy()

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=test_size,random_state=seed,stratify=y)



##
Ensemble 1 — Soft Voting
# Diverse base learners: only 2 tree-based out of 7
# Soft voting averages predict_proba → AUC/AUPRC computable


In [ ]:
# Build the base models
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier

# Step 1: Tune each base learner independently
# Note: LR, SVM, KNN, MLP require StandardScaler inside
#       their own pipelines so scaling never touches y or
#       leaks across folds.


def tune_lr_voter(X_train,y_train,seed=seed):
    pipe=Pipeline([
        ('scaler',StandardScaler()),
        ('lr',LogisticRegression(random_state=seed,max_iter=1000))
    ])
    param_grid = {
        'lr__C': [0.01, 0.1, 1, 10],
        'lr__penalty': ['l2'],
        'lr__class_weight': [None, 'balanced']
    }
    grid = GridSearchCV(pipe, param_grid,cv=5,
                        scoring='average_precision',n_jobs=-1)
    grid.fit(X_train,y_train)
    print(f"LR (voter) best params: {grid.best_params_}")
    return grid.best_estimator_


#scale-invariant — no scaler needed
def tune_random_forest(X_train, y_train, seed=42):
    """GridSearchCV for Random Forest"""
    param_grid = {
        'n_estimators': [10, 50, 100],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5]
    }

    grid = GridSearchCV(
        estimator=RandomForestClassifier(random_state=seed),
        param_grid=param_grid,
        cv=5,
        scoring='average_precision',
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)
    print(f"RF Best params: {grid.best_params_}")
    return grid.best_estimator_


#scale-invariant — no scaler needed
def tune_gradient_boosting(X_train, y_train, seed=42):
    """GridSearchCV for Gradient Boosting"""
    param_grid = {
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7]
    }

    grid = GridSearchCV(
        estimator=GradientBoostingClassifier(random_state=seed),
        param_grid=param_grid,
        cv=5,
        scoring='average_precision',
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)
    print(f"GB Best params: {grid.best_params_}")
    return grid.best_estimator_


def tune_mlp_voter(X_train, y_train, seed=42):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPClassifier(random_state=seed, max_iter=500,
                              early_stopping=True))
    ])
    param_grid = {
        'mlp__hidden_layer_sizes': [(64,), (64, 32), (128, 64)],
        'mlp__alpha': [0.0001, 0.001, 0.01],   # L2 regularization
        'mlp__learning_rate_init': [0.001, 0.01]
    }
    grid = GridSearchCV(pipe, param_grid, cv=5,
                        scoring='average_precision', n_jobs=-1)
    grid.fit(X_train, y_train)
    print(f"MLP (voter) best params: {grid.best_params_}")
    return grid.best_estimator_


def tune_svm(X_train, y_train, seed=42):
    """GridSearchCV for SVM"""
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', probability=True, random_state=seed))
    ])
    param_grid = {
        'svm__C': [0.1, 1, 10],
        'svm__gamma': ['scale', 'auto']
    }

    grid = GridSearchCV(
        pipe,param_grid,cv=5
        ,scoring='average_precision',
        n_jobs=-1
    )
    grid.fit(X_train,y_train)
    print(f"SVM Best params: {grid.best_params_}")
    return grid.best_estimator_

# NB has no meaningful hyperparameters for this feature set
def train_nb(X_train, y_train):
    """Train Naive Bayes (no tuning needed)"""
    nb = GaussianNB()
    nb.fit(X_train, y_train)
    return nb


def tune_knn(X_train, y_train, seed=42):
    """GridSearchCV for KNN"""
    pipe=Pipeline([
        ('scaler',StandardScaler()),
        ('knn',KNeighborsClassifier())
    ])
    param_grid = {
        'knn__n_neighbors': [3, 5, 7, 9, 11],
        'knn__weights': ['uniform', 'distance']
    }

    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=5,
        scoring='average_precision',
        n_jobs=-1,
    )

    grid.fit(X_train, y_train)
    print(f"KNN Best params: {grid.best_params_}")
    return grid.best_estimator_

from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC



In [ ]:
# Step 2: Fit all 7 tuned base learners
# All use scoring='average_precision' — consistent with LR
# baseline tuning objective
# ----------------------------------------------------------

print("Tuning soft voting base learners...")

voter_base_models = {
    'lgrg':  tune_lr_voter(X_train, y_train),
    'svm': tune_svm(X_train, y_train),
    'knn': tune_knn(X_train, y_train),
    'nb':  train_nb(X_train, y_train),
    'mlp': tune_mlp_voter(X_train, y_train),
    'rf':  tune_random_forest(X_train, y_train),   # your existing fn
    'gb':  tune_gradient_boosting(X_train, y_train) # your existing fn
}

soft_voting_ensembles = VotingClassifier(
    estimators=list(voter_base_models.items()),
    voting='soft',
    n_jobs=-1,
)

soft_voting_ensembles.fit(X_train,y_train)

In [ ]:
# Note for logistic regression, we pass scaled x_test
metrics_soft_voting = evaluate_model(
    soft_voting_ensembles, X_test, y_test, "Soft Voting Ensemble",threshold=prob_threshold
)

model_metrics_df = pd.concat(
    [model_metrics_df, pd.DataFrame([metrics_soft_voting])],
    ignore_index=True
)



In [ ]:
model_metrics_df

##
Ensemble 2 — Bagging (Bootstrap Aggregating)

Takes one base model — DecisionTree — and trains 100 copies of it, each on a different random sample of the training data. Each sample is drawn with replacement, meaning some patients appear multiple times and some not at all. This is called bootstrap sampling. The 100 trees then vote together. The key insight is that a single DecisionTree overfits badly — you saw Train=1.000 earlier. Bagging fixes this by averaging out the noise across 100 slightly different trees.

Imagine instead of 7 different doctors, you take one doctor and train them 100 times on slightly different subsets of patient records. Each time they see a slightly different version of the data. Then you average their answers.


pythonBaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100
)

The base model is always a DecisionTree. But each of the 100 trees sees a random sample of the training data — some patients appear twice, some not at all. This is called bootstrap sampling. The 100 trees then vote together.
The key benefit is reducing overfitting. A single DecisionTree overfits badly — you saw this in your train vs test plot where DTree hit 1.0 training accuracy. Bagging smooths this out by averaging 100 slightly different trees.


In [ ]:
#Ensemble 2 - Bagging (Bootstrap Aggregating)
from sklearn.ensemble import BaggingClassifier
bagging_ensemble = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=seed),
    n_estimators=100,
    random_state=seed,
    n_jobs=-1
)
#tune the bagging ensemble using GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_samples': [0.5, 0.75, 1.0],
    'max_features': [0.5, 0.75, 1.0]
}
grid_bagging = GridSearchCV(
    estimator=bagging_ensemble,
    param_grid=param_grid,
    cv=5,
    scoring='average_precision',
    n_jobs=-1,
    verbose=1
)
grid_bagging.fit(X_train, y_train)
print(f"Bagging Best params: {grid_bagging.best_params_}")
bagging_ensemble = grid_bagging.best_estimator_


### Observations:


-

In [ ]:
#Evaluate paper lr, tuned ensemble, and bagging ensemble
metrics_bagging =evaluate_model(bagging_ensemble, X_test, y_test, "Bagging Ensemble")
model_metrics_df = pd.concat(
    [model_metrics_df, pd.DataFrame([metrics_bagging])],
    ignore_index=True
)

In [ ]:
model_metrics_df

## Ensemble 3 — Stacking ( Meta -Learning)

Stacking takes the predictions of multiple base models and feeds them into a meta-model, which learns how to best combine those predictions. The base models can be any type (e.g., Random Forest, SVM, KNN), and the meta-model is often a simple model like Logistic Regression. The key insight is that different models capture different patterns in the data, and the meta-model learns to weigh their predictions to improve overall performance.

This is the most complex. Instead of a simple vote, stacking uses a second model to learn how to combine the predictions of the first models.



In [ ]:
#Ensemble 3 - Stacking (Meta-Learning)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
base_learners = [
    ('rf', tune_random_forest(X_train, y_train)),   # your existing fn
    ('gb', tune_gradient_boosting(X_train, y_train)),   # your existing fn
    ('svm', tune_svm(X_train, y_train)),   # your existing fn
    ('mlp', tune_mlp_voter(X_train,y_train)),
    ('nb', train_nb(X_train,y_train)),
    ('knn', tune_knn(X_train,y_train))
]
meta_learner = LogisticRegression(random_state=seed)
stacking_ensemble = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1,
    passthrough=False
)



In [ ]:
#tune the stacking ensemble using GridSearchCV
param_grid = {
    'final_estimator__C': [0.01, 0.1, 1, 10],
    'final_estimator__penalty': ['l2'],
    'final_estimator__solver': ['lbfgs']
}
grid_stacking = GridSearchCV(
    estimator=stacking_ensemble,
    param_grid=param_grid,
    cv=5,
    scoring='average_precision',
    n_jobs=-1,
    verbose=1
)
grid_stacking.fit(X_train, y_train)
print(f"Stacking Best params: {grid_stacking.best_params_}")
stacking_ensemble = grid_stacking.best_estimator_

## Observations:


-


In [ ]:
#Evaluate paper lr, tuned ensemble, bagging ensemble, and stacking ensemble
metrics_stacking =evaluate_model(stacking_ensemble, X_test, y_test, "Stacking Ensemble")

model_metrics_df = pd.concat(
    [model_metrics_df, pd.DataFrame([metrics_stacking])],
    ignore_index=True
)

In [ ]:
model_metrics_df

In [ ]:
# ============================================================
# Confusion Matrices — all models at threshold=0.35
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

models_to_plot = [
    (paper_tuned_lr,       X_test_paper_scaled, y_test_paper, "Paper LR (0.35)"),
    (soft_voting_ensembles, X_test,             y_test,       "Soft Voting (0.35)"),
    (bagging_ensemble,      X_test,             y_test,       "Bagging (0.35)"),
    (stacking_ensemble,     X_test,             y_test,       "Stacking (0.35)"),
]

for i, (model, X, y_true, title) in enumerate(models_to_plot):
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= prob_threshold).astype(int)   # prob_threshold=0.35

    cm = confusion_matrix(y_true, y_pred)

    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        ax=axes[i], cbar=False,
        xticklabels=['No DM', 'DM'],
        yticklabels=['No DM', 'DM']
    )

    acc  = accuracy_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)

    axes[i].set_title(f"{title}\nAcc={acc:.3f}  Rec={rec:.3f}  Prec={prec:.3f}",
                      fontsize=10)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

# hide unused subplot (2x3 grid, 4 models)
axes[4].set_visible(False)
axes[5].set_visible(False)

plt.suptitle(f'Confusion Matrices — threshold={prob_threshold}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
model_metrics_df

## Overall comparison of ensembles, paper logistic regression





In [ ]:
def find_best_clinical_model(df):
    # Normalize key metrics
    df_score = df.copy()

    # Fill NaNs to avoid issues
    df_score['AUPRC'] = df_score['AUPRC'].fillna(0)

    # Weighted score (adjust weights if needed)
    df_score['score'] = (
            0.5 * df_score['AUPRC'] +
            0.3 * df_score['Recall'] +
            0.2 * df_score['Precision']
    )
    print(df_score[['Model', 'score']].sort_values('score',ascending=False).to_string(index=False))
    winner = df_score.loc[df_score['score'].idxmax()]

    print("\n🏆 BEST CLINICAL MODEL")
    print(f"Model : {winner['Model']}")
    print(f"Score : {winner['score']:.3f}")

    return winner

## Justification:

1. AUPRC (Area Under Precision-Recall Curve) – Weight: 50%
Why It’s the Primary Metric (50% Weight)

Designed for imbalanced data: Unlike ROC-AUC, AUPRC focuses on the positive (minority) class (diabetic patients in this case).
Directly measures clinical utility:

High AUPRC = The model is good at ranking high-risk patients at the top of its predictions.
In screening, you want to prioritize patients for limited resources (e.g., doctor follow-ups). AUPRC tells you how well the model does this.

Robust to class imbalance: ROC-AUC can be misleading when classes are imbalanced (e.g., 65% non-diabetic vs. 35% diabetic). AUPRC is not fooled by imbalance.
Clinical Justification

A model with AUPRC = 0.7 means it’s 70% better than random at identifying diabetic patients among those it flags as high-risk.
Example: If a clinic can only follow up with 20% of patients, a high AUPRC ensures the 20% most at-risk patients are selected.

2. Recall (Sensitivity) – Weight: 30%
Why It’s Critical (30% Weight)

Recall = True Positives / (True Positives + False Negatives)
Measures the model’s ability to catch diabetic patients.
Missed diabetes cases (false negatives) are catastrophic in healthcare.
Clinical Justification

A recall of 90% means the model misses only 10% of diabetic patients.
In a population of 1000 patients with 35% diabetes prevalence (~350 diabetic patients):

Recall = 90% → 35 missed cases (10% of 350).
Recall = 70% → 105 missed cases (30% of 350).

35 vs. 105 missed diagnoses is the difference between acceptable and unacceptable for a screening tool.
Trade-off with Precision

Increasing recall always decreases precision (more false positives).
But in screening, false positives are tolerable if they reduce false negatives.

3. Precision – Weight: 20%
Why It’s Secondary (20% Weight)

Precision = True Positives / (True Positives + False Positives)
Measures the proportion of predicted diabetic patients who are actually diabetic.
Lower weight because false positives are less harmful than false negatives in screening.
Clinical Justification

A precision of 60% means 40% of flagged patients are not diabetic.
While this increases follow-up costs, it’s preferable to missing 30% of diabetic patients (as in the original LR model).
False positives can be filtered out with confirmatory tests (e.g., HbA1c).



In [ ]:
winning_model = find_best_clinical_model(model_metrics_df)

## Justification:

AUPRC (Area under PR Curve ) - 50%
this is a primary metric designed for imbalanceed data- unlike ROC-AUC, AUPRC focuses on the positive (minority class)
Directly measures clinical utility
High AUPRC means good at ranking high-risk patients at the top of its predictions

Clinical Justification

A model with AUPRC = 0.7 means it’s 70% better than random at identifying diabetic patients among those it flags as high-risk.
Example: If a clinic can only follow up with 20% of patients, a high AUPRC ensures the 20% most at-risk patients are selected.


Recall - 30%
Clinical Justification

A recall of 90% means the model misses only 10% of diabetic patients.
In a population of 1000 patients with 35% diabetes prevalence (~350 diabetic patients):

Recall = 90% → 35 missed cases (10% of 350).
Recall = 70% → 105 missed cases (30% of 350).

35 vs. 105 missed diagnoses is the difference between acceptable and unacceptable for a screening tool.

Remaining is Precision at 20% which is a secondary metric




In [ ]:
# Final thoughts:
# - The tuned ensembl

In [ ]:
## Save the best model for future use
import pickle
import os
os.makedirs('models', exist_ok=True)
best_model_name = winning_model['Model'].replace(" ", "_").lower()
with open(f'models/{best_model_name}.pkl', 'wb') as f:
    if winning_model['Model'] == "Logistic Regression (Paper)":
        pickle.dump(paper_tuned_lr, f)
    elif winning_model['Model'] == "Tuned Ensemble":
        pickle.dump(soft_voting_ensembles, f)
    elif winning_model['Model'] == "Bagging Ensemble":
        pickle.dump(bagging_ensemble, f)
    elif winning_model['Model'] == "Stacking Ensemble":
        pickle.dump(stacking_ensemble, f)
print(f"\nBest model '{winning_model['Model']}' saved to 'models/{best_model_name}.pkl'")





## Model Serving

call the diabetic_serving.py script using below command. (remember to run from same location as file)

python -m streamlit run .\diabetic_serving.py
